# Step-edge filter walkthrough

This notebook creates a `1024 x 1024` image whose left half is black and whose right half is white. The examples run several filter and execution-path combinations so a new user can see the basic library pattern and compare the center-row response at the vertical edge.

The final figure uses standard notebook plotting with `matplotlib`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:

    def HTML(value):
        return value

    def display(value):
        print(value)


repo_root = Path.cwd()
if not (repo_root / "agfb_filters").exists() and (repo_root.parent / "agfb_filters").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agfb_filters import (  # noqa: E402
    BoundaryCondition,
    BoundaryMode,
    ExecutionPath,
    collapse_orientation_bank,
    get_filter_definition,
    run_filter,
    run_orientation_bank,
)

torch.set_grad_enabled(False)
plt.rcParams["figure.dpi"] = 120
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

The image uses the package convention `(batch, height, width)`. The edge is at column `512`, so `gradient_x` should carry the response and `gradient_y` should stay near zero away from boundary artifacts.

In [ ]:
height = 1024
width = 1024
edge_column = width // 2

image = torch.zeros(1, height, width, dtype=torch.float32, device=device)
image[:, :, edge_column:] = 1.0

image.shape, float(image.min()), float(image.max()), edge_column

Each run below shows the same call shape. Get a definition, choose an explicit execution path, choose a boundary, then call `run_filter`. The orientation-bank example uses `run_orientation_bank` first and collapses the raw bank only for this comparison plot.

In [ ]:
profile_radius = 40
profile_offsets = list(range(-profile_radius, profile_radius + 1))

runs = [
    {
        "label": "central difference",
        "definition": get_filter_definition("central_difference"),
        "path": ExecutionPath.SEPARABLE,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
    {
        "label": "sobel 3",
        "definition": get_filter_definition("sobel_3"),
        "path": ExecutionPath.SEPARABLE,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
    {
        "label": "farid simoncelli 7",
        "definition": get_filter_definition("farid_simoncelli_7"),
        "path": ExecutionPath.SEPARABLE,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
    {
        "label": "sparse central r=4",
        "definition": get_filter_definition("sparse_central_difference", radius=4),
        "path": ExecutionPath.SPARSE_OFFSETS,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
    {
        "label": "haar box r=6",
        "definition": get_filter_definition("haar_box_gradient", radius=6),
        "path": ExecutionPath.BOX_INTEGRAL,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
    {
        "label": "perona malik",
        "definition": get_filter_definition(
            "perona_malik_gradient",
            iterations=5,
            step_size=0.15,
            kappa=0.2,
        ),
        "path": ExecutionPath.ITERATIVE,
        "boundary": BoundaryCondition(BoundaryMode.REPLICATE),
    },
]

profiles = []
summary_rows = []

for run in runs:
    gradient_x, gradient_y = run_filter(
        run["definition"],
        image,
        path=run["path"],
        boundary=run["boundary"],
    )
    center_row = gradient_x.shape[-2] // 2
    profile = (
        gradient_x[
            0,
            center_row,
            edge_column - profile_radius : edge_column + profile_radius + 1,
        ]
        .detach()
        .cpu()
    )
    profiles.append({"label": run["label"], "values": profile.tolist()})
    summary_rows.append(
        {
            "filter": run["label"],
            "path": run["path"].value,
            "boundary": run["boundary"].mode.value,
            "max_abs_gx": float(gradient_x.abs().max().detach().cpu()),
            "max_abs_gy": float(gradient_y.abs().max().detach().cpu()),
            "edge_left": float(profile[profile_radius - 1]),
            "edge_right": float(profile[profile_radius]),
        }
    )

bank_definition = get_filter_definition(
    "anisotropic_gaussian_orientation_bank",
    angle_count=8,
    sigma_parallel=1.0,
    sigma_perpendicular=2.0,
)
bank = run_orientation_bank(
    bank_definition,
    image,
    path=ExecutionPath.ORIENTATION_BANK,
    boundary=bank_definition.default_boundary,
)
collapsed_bank = collapse_orientation_bank(bank, mode="least_squares_projection")
bank_profile = (
    collapsed_bank.gradient_x[
        0,
        height // 2,
        edge_column - profile_radius : edge_column + profile_radius + 1,
    ]
    .detach()
    .cpu()
)
profiles.append({"label": "orientation bank ls", "values": bank_profile.tolist()})
summary_rows.append(
    {
        "filter": "orientation bank ls",
        "path": ExecutionPath.ORIENTATION_BANK.value,
        "boundary": bank_definition.default_boundary.mode.value,
        "max_abs_gx": float(collapsed_bank.gradient_x.abs().max().detach().cpu()),
        "max_abs_gy": float(collapsed_bank.gradient_y.abs().max().detach().cpu()),
        "edge_left": float(bank_profile[profile_radius - 1]),
        "edge_right": float(bank_profile[profile_radius]),
    }
)

len(profiles), summary_rows[:2]

The robust local plane filter solves a weighted least-squares problem at each pixel. This cell runs it on a centered crop from the same `1024 x 1024` image so the example stays quick while still showing the call pattern.

In [ ]:
crop_radius = 128
crop = image[
    :,
    height // 2 - crop_radius : height // 2 + crop_radius,
    edge_column - crop_radius : edge_column + crop_radius,
]
crop_edge_column = crop.shape[-1] // 2

local_plane_definition = get_filter_definition(
    "robust_local_plane_gradient",
    radius=2,
    weighting="huber",
    robust_scale=0.5,
)
local_x, local_y = run_filter(
    local_plane_definition,
    crop,
    path=ExecutionPath.NONLINEAR_WINDOW,
    boundary=local_plane_definition.default_boundary,
)
local_profile = (
    local_x[
        0,
        crop.shape[-2] // 2,
        crop_edge_column - profile_radius : crop_edge_column + profile_radius + 1,
    ]
    .detach()
    .cpu()
)
profiles.append({"label": "local plane crop", "values": local_profile.tolist()})
summary_rows.append(
    {
        "filter": "local plane crop",
        "path": ExecutionPath.NONLINEAR_WINDOW.value,
        "boundary": local_plane_definition.default_boundary.mode.value,
        "max_abs_gx": float(local_x.abs().max().detach().cpu()),
        "max_abs_gy": float(local_y.abs().max().detach().cpu()),
        "edge_left": float(local_profile[profile_radius - 1]),
        "edge_right": float(local_profile[profile_radius]),
    }
)

crop.shape, len(profiles)

The summary table is useful for checking which axis responded and how each path was called.

In [ ]:
def display_summary(rows):
    header = "".join(
        f"<th>{name}</th>"
        for name in ("filter", "path", "boundary", "max |gx|", "max |gy|", "left", "right")
    )
    body = []
    for row in rows:
        body.append(
            "<tr>"
            f"<td>{row['filter']}</td>"
            f"<td><code>{row['path']}</code></td>"
            f"<td><code>{row['boundary']}</code></td>"
            f"<td>{row['max_abs_gx']:.6f}</td>"
            f"<td>{row['max_abs_gy']:.6f}</td>"
            f"<td>{row['edge_left']:.6f}</td>"
            f"<td>{row['edge_right']:.6f}</td>"
            "</tr>"
        )
    display(
        HTML(
            "<table>"
            "<thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"
        )
    )


display_summary(summary_rows)

The plot shows the input step edge on the left and the center-row `gradient_x` response on the right. The profiles are sampled from column offsets `-40` through `+40` around the split.

In [ ]:
brand_colors = [
    "#73000A",
    "#466A9F",
    "#1F414D",
    "#65780B",
    "#CC2E40",
    "#A49137",
    "#000000",
    "#5C5C5C",
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].imshow(
    image[0].detach().cpu().numpy(),
    cmap="gray",
    vmin=0.0,
    vmax=1.0,
    interpolation="nearest",
)
axes[0].axvline(edge_column - 0.5, color="#73000A", linewidth=1.5)
axes[0].set_title("1024 x 1024 step image")
axes[0].set_axis_off()

for index, profile in enumerate(profiles):
    axes[1].plot(
        profile_offsets,
        profile["values"],
        label=profile["label"],
        color=brand_colors[index % len(brand_colors)],
        linewidth=1.8,
    )

axes[1].axvline(0, color="#73000A", linewidth=1.0, linestyle="--", label="edge")
axes[1].axhline(0, color="#C7C7C7", linewidth=0.8)
axes[1].set_title("Center-row gradient_x response")
axes[1].set_xlabel("column offset from edge")
axes[1].set_ylabel("response")
axes[1].grid(True, color="#ECECEC", linewidth=0.8)
axes[1].legend(loc="upper right", fontsize=8)

fig